In [9]:
import pandas as pd
import numpy as np
from db_queries import get_covariate_estimates, get_location_metadata

Load data paths

In [3]:
data_path = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/training_data/2026_09_13.01/data.parquet"
no_cutoff_path = "/mnt/team/rapidresponse/pub/population/modeling/climate_malnutrition/neonatal_mortality/training_data/2026_09_13.01/data_no_10_yr_cutoff.parquet"

In [15]:
df = pd.read_parquet(data_path)
df.drop(columns=["index"], inplace=True)
df.head()

,year_start,year_end,nid,survey_name,int_year,int_month,sex_id,mothers_age_year,aod_months,age_month,...,q95_prev_6_mo_avg,q95_prev_9_mo_avg,q99_prev_3_mo_avg,q99_prev_6_mo_avg,q99_prev_9_mo_avg,zone,year_id,age_group_id,int_birth_year_diff_months,log_consumption_pd
0,2009,2010,13109,DHS_MIS,None,None,2,None,0,0,...,0.000000,0.666667,0.000000,0.000000,0.000000,23.0,2009,42,0,-0.245692
1,2022,2022,538795,None,2022,5,2,18,0,0,...,2.333333,1.777778,0.000000,0.833333,0.555556,24.0,2021,42,11,1.312609
2,2017,2017,353526,None,2017,4,1,20,0,0,...,5.000000,3.444444,3.666667,1.833333,1.222222,27.0,2017,42,0,2.448691
3,2019,2021,467681,DHS,None,None,1,None,0,0,...,0.000000,0.777778,0.000000,0.000000,0.000000,24.0,2020,42,2,1.619791
4,2000,2000,19156,None,2000,2,2,24,0,0,...,0.000000,0.000000,0.000000,0.000000,0.000000,25.0,1999,42,2,-0.227447


Check for missing values and max differences between birth years and interview years

In [ ]:
cols_to_check = [
    "neonatal_mortality",
    "consumption_pd",
    "int_birth_year_diff_months",
    "birth_year",
    "days_over_30C_prev_0_mo",
    "total_precipitation_prev_0_mo",
]

for c in cols_to_check:
    print(f"{c}: {df[c].isna().sum()} missing values")

neonatal_mortality: 0 missing values
consumption_pd: 0 missing values
int_birth_year_diff_months: 0 missing values
birth_year: 0 missing values
days_over_30C_prev_0_mo: 0 missing values
total_precipitation_prev_0_mo: 0 missing values


In [7]:
len(df)

6647813

Cut off int_birth_year_diff_months to max 5

In [15]:
df.to_parquet(no_cutoff_path, index=False)

In [16]:
df = df[df["int_birth_year_diff_months"] < 60]
print(f"Number of rows after filtering: {len(df):,}")
df.to_parquet(data_path, index=False)

Number of rows after filtering: 1,711,538


Create log-consumption_pd, making sure there is an offset from true zero

In [4]:
df["consumption_pd"].min()  # already non-zero, ~ $0.0232/year

6.359990070799202e-05

In [5]:
sorted([c for c in df.columns])

['age_group_id',
 'age_month',
 'age_month_original',
 'any_days_over_30C',
 'aod_months',
 'birth_month',
 'birth_year',
 'child_alive',
 'child_mortality',
 'consumption',
 'consumption_pd',
 'cum_weight',
 'days_over_24C_prev_0_mo',
 'days_over_24C_prev_1_mo',
 'days_over_24C_prev_2_mo',
 'days_over_24C_prev_3_mo',
 'days_over_24C_prev_3_mo_avg',
 'days_over_24C_prev_4_mo',
 'days_over_24C_prev_5_mo',
 'days_over_24C_prev_6_mo',
 'days_over_24C_prev_6_mo_avg',
 'days_over_24C_prev_7_mo',
 'days_over_24C_prev_8_mo',
 'days_over_24C_prev_9_mo',
 'days_over_24C_prev_9_mo_avg',
 'days_over_25C_prev_0_mo',
 'days_over_25C_prev_1_mo',
 'days_over_25C_prev_2_mo',
 'days_over_25C_prev_3_mo',
 'days_over_25C_prev_3_mo_avg',
 'days_over_25C_prev_4_mo',
 'days_over_25C_prev_5_mo',
 'days_over_25C_prev_6_mo',
 'days_over_25C_prev_6_mo_avg',
 'days_over_25C_prev_7_mo',
 'days_over_25C_prev_8_mo',
 'days_over_25C_prev_9_mo',
 'days_over_25C_prev_9_mo_avg',
 'days_over_26C_prev_0_mo',
 'days_over_

In [6]:
df["log_consumption_pd"] = np.log(df["consumption_pd"])
df["log_consumption_pd"].head()

0   -0.245692
1    1.312609
2    2.448691
3    1.619791
4   -0.227447
Name: log_consumption_pd, dtype: float64

In [7]:
df.to_parquet(data_path, index=False)

9/16 - add SDI for experiment

In [5]:
sdi = get_covariate_estimates(
    covariate_id=881,  # SDI
    release_id=16,  # GBD 2023
    # location_id=<your_locations>,
    # year_id=list(range(1980, 2024)),  # see years note below
)

In [ ]:
sdi.rename(columns={"year_id": "birth_year", "mean_value": "sdi"}, inplace=True)
sdi = sdi[["birth_year", "location_id", "sdi"]].drop_duplicates(ignore_index=True)

In [16]:
df = df.merge(sdi, on=["birth_year", "location_id"], how="left")
df.head()

,year_start,year_end,nid,survey_name,int_year,int_month,sex_id,mothers_age_year,aod_months,age_month,...,q95_prev_9_mo_avg,q99_prev_3_mo_avg,q99_prev_6_mo_avg,q99_prev_9_mo_avg,zone,year_id,age_group_id,int_birth_year_diff_months,log_consumption_pd,sdi
0,2009,2010,13109,DHS_MIS,None,None,2,None,0,0,...,0.666667,0.000000,0.000000,0.000000,23.0,2009,42,0,-0.245692,0.318291
1,2022,2022,538795,None,2022,5,2,18,0,0,...,1.777778,0.000000,0.833333,0.555556,24.0,2021,42,11,1.312609,0.447416
2,2017,2017,353526,None,2017,4,1,20,0,0,...,3.444444,3.666667,1.833333,1.222222,27.0,2017,42,0,2.448691,0.373539
3,2019,2021,467681,DHS,None,None,1,None,0,0,...,0.777778,0.000000,0.000000,0.000000,24.0,2020,42,2,1.619791,0.583046
4,2000,2000,19156,None,2000,2,2,24,0,0,...,0.000000,0.000000,0.000000,0.000000,25.0,1999,42,2,-0.227447,0.340183


In [17]:
df.to_parquet(data_path, index=False)